# 02 — Rating-Volatilität

Frage: *Schwanken die Ratings in einer Gruppe stärker als in der anderen?*

Kennzahl: `normalized_volatility` = Ø |rating_change| geteilt durch K-Faktor — das macht Spieler mit unterschiedlichem K vergleichbar.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
from _setup import load_view, load_query, apply_style, GROUP_PALETTE, GROUP_ORDER

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

apply_style()

## Daten laden

In [ ]:
df = load_view('v_rating_volatility')
df['period'] = pd.to_datetime(df['period'])
df[['avg_abs_change', 'normalized_volatility']] = (
    df[['avg_abs_change', 'normalized_volatility']].astype(float)
)
df = df.dropna(subset=['normalized_volatility'])
print(df.shape)
df.head()

## Summary pro Gruppe

In [ ]:
df.groupby('analysis_group').agg(
    n=('fide_id', 'count'),
    mean_abs_change=('avg_abs_change', 'mean'),
    mean_norm_vol=('normalized_volatility', 'mean'),
    median_norm_vol=('normalized_volatility', 'median'),
).round(3)

## Boxplot — normalisierte Volatilität

In [ ]:
fig, ax = plt.subplots()
sns.boxplot(
    data=df, x='analysis_group', y='normalized_volatility',
    order=GROUP_ORDER, palette=GROUP_PALETTE, ax=ax,
)
ax.set_ylabel('Ø |rating_change| / K-Faktor')
ax.set_xlabel('')
ax.set_title('Normalisierte Rating-Volatilität pro (Spieler, Periode)')
plt.tight_layout(); plt.show()

## Volatilität über Zeit

Ø normalisierte Volatilität pro Periode und Gruppe.

In [ ]:
ts = df.groupby(['analysis_group', 'period'])['normalized_volatility'].mean().reset_index()
fig, ax = plt.subplots()
sns.lineplot(
    data=ts, x='period', y='normalized_volatility',
    hue='analysis_group', hue_order=GROUP_ORDER, palette=GROUP_PALETTE, ax=ax,
)
ax.set_ylabel('Ø normalisierte Volatilität')
ax.set_title('Rating-Volatilität über Zeit')
plt.tight_layout(); plt.show()